# Estimation Theory - Examples

Practical demonstrations of statistical estimation for ML.

## Topics Covered
1. Bias-Variance Tradeoff
2. Maximum Likelihood Estimation (MLE)
3. MLE for Common Distributions
4. Method of Moments
5. Confidence Intervals

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize

plt.style.use('seaborn-v0_8-whitegrid')
np.set_printoptions(precision=6, suppress=True)

## 1. Bias-Variance Tradeoff

**Bias**: $E[\hat{\theta}] - \theta$ (systematic error)

**Variance**: $E[(\hat{\theta} - E[\hat{\theta}])^2]$ (spread)

**MSE**: $Bias^2 + Variance$

In [ ]:
print("BIAS-VARIANCE TRADEOFF")
print("="*60)

np.random.seed(42)
true_mean = 100
n_samples = 20
n_simulations = 10000

# Simulate many estimates
sample_means = []
sample_vars_biased = []    # Divide by n
sample_vars_unbiased = []  # Divide by n-1

for _ in range(n_simulations):
    sample = np.random.normal(true_mean, 15, n_samples)
    sample_means.append(sample.mean())
    sample_vars_biased.append(sample.var(ddof=0))    # Biased
    sample_vars_unbiased.append(sample.var(ddof=1))  # Unbiased

sample_means = np.array(sample_means)
sample_vars_biased = np.array(sample_vars_biased)
sample_vars_unbiased = np.array(sample_vars_unbiased)

true_var = 15**2  # 225

print("\n📊 Sample Mean Estimator:")
print(f"  E[X̄] = {sample_means.mean():.4f} (True μ = {true_mean})")
print(f"  Bias = {sample_means.mean() - true_mean:.6f}")
print(f"  Variance = {sample_means.var():.4f}")
print("  ✅ Sample mean is UNBIASED")

print("\n📊 Sample Variance (biased, ddof=0):")
print(f"  E[s²] = {sample_vars_biased.mean():.4f} (True σ² = {true_var})")
print(f"  Bias = {sample_vars_biased.mean() - true_var:.4f}")
print("  ❌ Underestimates true variance")

print("\n📊 Sample Variance (unbiased, ddof=1):")
print(f"  E[s²] = {sample_vars_unbiased.mean():.4f} (True σ² = {true_var})")
print(f"  Bias = {sample_vars_unbiased.mean() - true_var:.4f}")
print("  ✅ Corrected with Bessel's correction (n-1)")

## 2. Maximum Likelihood Estimation (MLE)

**Likelihood**: $L(\theta|x) = \prod_{i=1}^n f(x_i|\theta)$

**Log-Likelihood**: $\ell(\theta) = \sum_{i=1}^n \log f(x_i|\theta)$

**MLE**: $\hat{\theta}_{MLE} = \arg\max_\theta \ell(\theta)$

In [ ]:
print("MLE FOR NORMAL DISTRIBUTION")
print("="*60)

np.random.seed(42)
true_mu, true_sigma = 50, 10
data = np.random.normal(true_mu, true_sigma, 100)

print(f"\nTrue parameters: μ={true_mu}, σ={true_sigma}")
print(f"Sample size: n={len(data)}")

# Analytical MLE
mle_mu = data.mean()
mle_sigma = data.std(ddof=0)  # MLE uses n, not n-1

print(f"\n📊 MLE estimates:")
print(f"  μ̂_MLE = X̄ = {mle_mu:.4f}")
print(f"  σ̂_MLE = √(Σ(x-x̄)²/n) = {mle_sigma:.4f}")

# Visualize likelihood as function of μ
def neg_log_likelihood_mu(mu, data, sigma):
    return -np.sum(stats.norm.logpdf(data, mu, sigma))

mu_range = np.linspace(45, 55, 100)
nll_values = [neg_log_likelihood_mu(mu, data, true_sigma) for mu in mu_range]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(mu_range, nll_values, 'b-', linewidth=2)
ax.axvline(mle_mu, color='r', linestyle='--', label=f'MLE μ̂={mle_mu:.2f}')
ax.axvline(true_mu, color='g', linestyle=':', label=f'True μ={true_mu}')
ax.set_xlabel('μ')
ax.set_ylabel('Negative Log-Likelihood')
ax.set_title('Finding MLE: Minimize Negative Log-Likelihood')
ax.legend()
plt.show()

print("\n💡 MLE finds the peak of the likelihood (valley of neg-log-likelihood)")

## 3. MLE for Common Distributions

In [ ]:
print("MLE FOR COMMON DISTRIBUTIONS")
print("="*60)

np.random.seed(42)

# Bernoulli/Binomial
print("\n🎲 BERNOULLI (coin flips):")
true_p = 0.7
flips = np.random.binomial(1, true_p, 100)  # 100 coin flips
mle_p = flips.mean()
print(f"  True p = {true_p}")
print(f"  MLE p̂ = (# successes / n) = {mle_p:.4f}")

# Poisson
print("\n📊 POISSON (count data):")
true_lambda = 5
counts = np.random.poisson(true_lambda, 100)
mle_lambda = counts.mean()
print(f"  True λ = {true_lambda}")
print(f"  MLE λ̂ = X̄ = {mle_lambda:.4f}")

# Exponential
print("\n⏱️ EXPONENTIAL (waiting times):")
true_rate = 2.0  # λ = rate
wait_times = np.random.exponential(1/true_rate, 100)  # scale = 1/rate
mle_rate = 1 / wait_times.mean()
print(f"  True λ = {true_rate}")
print(f"  MLE λ̂ = 1/X̄ = {mle_rate:.4f}")

print("\n💡 MLE formulas are derived by setting d/dθ[log L] = 0")

## 4. Method of Moments

Match sample moments to population moments:
- First moment: $E[X] = \bar{X}$
- Second moment: $E[X^2] = \frac{1}{n}\sum X_i^2$

In [ ]:
print("METHOD OF MOMENTS")
print("="*60)

np.random.seed(42)

# Gamma distribution example
# Gamma(α, β): E[X] = α/β, Var[X] = α/β²
true_alpha, true_beta = 3, 2
data = np.random.gamma(true_alpha, 1/true_beta, 500)

print(f"\nTrue parameters: α={true_alpha}, β={true_beta}")

# Sample moments
m1 = data.mean()       # First moment
m2 = (data**2).mean()  # Second moment
sample_var = data.var()

print(f"\nSample moments:")
print(f"  m₁ = E[X] = {m1:.4f}")
print(f"  m₂ = E[X²] = {m2:.4f}")
print(f"  Var = m₂ - m₁² = {sample_var:.4f}")

# Method of Moments estimates
# From E[X] = α/β and Var[X] = α/β²:
# β = E[X]/Var[X]
# α = E[X] × β
mm_beta = m1 / sample_var
mm_alpha = m1 * mm_beta

print(f"\n📊 Method of Moments estimates:")
print(f"  α̂ = {mm_alpha:.4f} (True: {true_alpha})")
print(f"  β̂ = {mm_beta:.4f} (True: {true_beta})")

# Compare with MLE (scipy fit)
mle_alpha, _, mle_scale = stats.gamma.fit(data, floc=0)
mle_beta = 1/mle_scale

print(f"\n📊 MLE estimates:")
print(f"  α̂ = {mle_alpha:.4f}")
print(f"  β̂ = {mle_beta:.4f}")

print("\n💡 MoM is simpler but MLE is generally more efficient")

## 5. Confidence Intervals

**For large n**: $\bar{X} \pm z_{\alpha/2} \frac{s}{\sqrt{n}}$

**For small n**: $\bar{X} \pm t_{\alpha/2, n-1} \frac{s}{\sqrt{n}}$

In [ ]:
print("CONFIDENCE INTERVALS")
print("="*60)

np.random.seed(42)
true_mean = 100
data = np.random.normal(true_mean, 15, 30)

n = len(data)
x_bar = data.mean()
s = data.std(ddof=1)
se = s / np.sqrt(n)

print(f"\nSample: n={n}, X̄={x_bar:.2f}, s={s:.2f}")
print(f"Standard Error: SE = s/√n = {se:.4f}")

# 95% CI using t-distribution
alpha = 0.05
t_crit = stats.t.ppf(1 - alpha/2, df=n-1)
margin = t_crit * se
ci_lower = x_bar - margin
ci_upper = x_bar + margin

print(f"\n95% Confidence Interval:")
print(f"  t-critical (α=0.05, df={n-1}): {t_crit:.4f}")
print(f"  Margin of Error: {margin:.4f}")
print(f"  CI: [{ci_lower:.2f}, {ci_upper:.2f}]")
print(f"  True mean ({true_mean}) in CI: {ci_lower <= true_mean <= ci_upper}")

# Coverage simulation
n_simulations = 1000
n_covered = 0
for _ in range(n_simulations):
    sample = np.random.normal(true_mean, 15, 30)
    x_bar = sample.mean()
    se = sample.std(ddof=1) / np.sqrt(len(sample))
    lower = x_bar - t_crit * se
    upper = x_bar + t_crit * se
    if lower <= true_mean <= upper:
        n_covered += 1

print(f"\n📊 Coverage Rate: {n_covered/n_simulations*100:.1f}% (should be ~95%)")

## 6. Fisher Information and Efficiency

In [ ]:
print("FISHER INFORMATION")
print("="*60)

np.random.seed(42)

# For Normal(μ, σ²) with known σ:
# Fisher Information I(μ) = n/σ²
# Cramér-Rao Lower Bound: Var(μ̂) ≥ 1/I(μ) = σ²/n

true_sigma = 10
sample_sizes = [10, 50, 100, 500]

print(f"\nFor Normal(μ, σ²={true_sigma**2}) with known σ:")
print(f"\n{'n':>6} {'Fisher I':>12} {'CRLB':>12} {'Theory SE':>12}")
print("-"*50)

for n in sample_sizes:
    fisher_info = n / true_sigma**2
    crlb = 1 / fisher_info  # Variance lower bound
    se_bound = np.sqrt(crlb)  # Standard error lower bound
    print(f"{n:>6} {fisher_info:>12.4f} {crlb:>12.6f} {se_bound:>12.6f}")

print("\n💡 More data → Higher Fisher Info → Lower variance bound")
print("💡 MLE achieves the CRLB asymptotically (efficient estimator)")